# 第九章：把同一个研究候选交到别人手里

前面各章分别讨论市场、数据、信号、统计、组合与模型。真正的研究交付要求它们在同一笔交易上相互吻合：你声称利用的信息当时是否可见，代码是否执行了那条规则，收益是否付过费用，结论是否超出了证据。本章只追踪一个候选：“过去五个交易日的日内平均收益为正时，下一日开盘做多，收盘平仓。”它足够简单，让我们把精力放在研究判断、失败定位和沟通上。

读完后，你应能把模糊想法压成可证伪的研究合同；从原始数据重建特征、仓位、成交与收益；把同一候选与合理基准、成本和延迟压力放在一起；形成一份包含负面证据的研究备忘录；最后用九十秒向不同岗位说明结果与边界。代码使用 NumPy 和标准库，默认不读外部文件、不联网，也不产生运行文件。

我们使用一条带有微弱延续关系的合成日内收益序列。它不是指数行情，不是可以交易的产品，也没有真实订单簿。实验的作用是演练一条可复核的研究链，而不是给出投资建议。这个区别会贯穿本章：能复现程序，不等于已经证明经济机制；回测完整，也不等于已经具备生产条件。


## 1. 把想法写成能被否定的一句话

“趋势策略能赚钱吗？”无法直接做实验，因为资产、持有期限、入场方式和比较对象都不明确。我们将它压成：“对一个可交易资产，如果短期日内走势存在延续，过去五日平均日内收益为正后，下一日日内做多的成本后结果，应该优于每天日内做多的基准；这一判断在合理成本与一日额外延迟下不能完全依赖单个时期。”

这里的**假设（hypothesis）**提出待检验的关系；**机制（mechanism）**给出可能解释，例如分散交易需求缓慢完成，价格有短期延续。机制不是结论，也不是看完收益曲线后补的故事。**资产宇宙（universe）**是可进入研究的资产集合，本例仅一个资产；真实跨资产项目必须明确入选与退出条件，不能只选现在还存在且表现好的品种。

**期限（horizon）**是信号到评价结束的时间跨度，本例为下一交易日日内。**基准（benchmark）**每天开盘做多、收盘平仓，与候选共享资产、可见价格和持有规则，能够判断择时是否优于持续参与。仅与不交易相比，可能把市场整体偏涨误认为信号有效；仅与全天买入持有相比，又会混入隔夜暴露差异。

本例事先固定五日窗口、只做多或空仓、无杠杆、每次进出各 1 bp，训练/验证/测试按时间顺序划分。主指标为测试期相对基准的平均每日净收益，另报回撤、参与比例和压力结果。这份**研究合同（research contract）**限制研究自由度：看完测试以后，不能把窗口换成表现最好的天数，再保留原测试标签。

一个有用的停止条件也应事先写下。例如若信号结果主要由少数事件贡献，或实际成本高于毛收益空间，就不进入资金部署；若只是统计证据不足，可以收集独立数据，而不必宣称机制已经被推翻。停止与继续都要指向具体证据，不能只说“感觉值得”。


In [ ]:
from dataclasses import dataclass, asdict
import hashlib
import json
import numpy as np

@dataclass(frozen=True)
class ResearchContract:
    universe: str = "一个合成日内资产"
    feature: str = "截至决策日收盘的过去五日日内收益均值"
    decision: str = "正值时次日开盘做多一单位，否则空仓；次日收盘平仓"
    benchmark: str = "每天开盘做多一单位，当日收盘平仓"
    window: int = 5
    cost_per_side: float = 0.0001
    train_end: int = 220
    validation_end: int = 320
    seed: int = 9

contract = ResearchContract()
print(json.dumps(asdict(contract), ensure_ascii=False, indent=2))


## 2. 六层管线不是六个文件夹，而是六个可解释对象

**管线（pipeline）**表示一个对象怎样成为下一个对象。数据层保存价格、收益与时间；特征层只使用当时可见的历史；信号层输出方向或预测；组合层决定允许持有多少；执行层把目标转成订单与成交；报告层解释收益和风险。每层可以很短，但边界不能含糊。

在本例中，数据是每日开盘到收盘收益；特征是五日均值；信号是均值是否大于零；组合权重限于 0 或 1；执行是次日开盘进、收盘出；报告比较扣费后的结果。信号为 1 并不等于已经买到资产，目标仓位为 1 也不等于一定全部成交。最小实现假设足额成交，交付时必须把这个假设显式写出来。




![研究管线五层与每层的检查门](figures/ch09_pipeline_layers.png)

**图 9-1｜六层结构，每层一个检查门。** 六个环节依次是数据、特征、信号、组合、执行和报告；箭头按处理方向向下。每一步都应有明确输入输出与检查条件。数据类型：研究工作流概念示意。


每层都能对应一句失败诊断。行情重复导致收益异常，是数据问题；五日均值误读未来，是特征问题；仓位多移一天，是时间映射问题；负仓位突破限制，是组合问题；漏收平仓费用，是执行问题；把训练表现写成样本外，是报告问题。把错误定位到具体对象，修复后才能知道哪些结果需要重算。

一笔历史交易至少要回答五个问题：信息何时可见、信号何时完成、订单何时发送、按何价多少量成交、费用与收益何时记账。本例决策在第 \(t\) 日收盘，目标作用在第 \(t+1\) 日日内；没有隔夜持仓，也不赚收盘到次日开盘的跳空。没有这条时间线，一个看起来只有两行的向量化回测也可能完全错位。


In [ ]:
rng = np.random.default_rng(contract.seed)
n_days = 420
intraday = np.zeros(n_days)
for day in range(1, n_days):
    coefficient = 0.14 if day < 300 else 0.0
    intraday[day] = coefficient * intraday[day - 1] + rng.normal(0.0001, 0.009)

# 每行按“实际交易日”存放，signal_day 是其前一日。
trade_day = np.arange(contract.window, n_days)
signal_day = trade_day - 1
feature = np.array([intraday[t-contract.window+1:t+1].mean() for t in signal_day])
weight = (feature > 0).astype(float)
actual = intraday[trade_day]
benchmark_weight = np.ones(len(trade_day))

assert (signal_day < trade_day).all()
assert np.isin(weight, [0.0, 1.0]).all()
print("交易行数:", len(trade_day))
for i in range(5):
    print("决策日", signal_day[i], "交易日", trade_day[i],
          "五日均值 bp", round(feature[i]*10000, 2), "仓位", weight[i])


## 3. 先手算一笔，再相信整条曲线

假设五个已完成交易日日内收益为 +20、-10、+15、-5、+10 bp，平均为 +6 bp，所以明天做多一单位。若明日日内收益 -30 bp，毛收益 -30 bp；开仓、平仓各 1 bp，净收益 -32 bp。若五日均值为负，则保持空仓，当天收益为零，不支付这笔交易的费用。不能在看到明天亏损后把那一行改为空仓。

本例的收益核算为

\[
g_d=w_dr_d,\qquad v_d=2|w_d|,\qquad q_d=g_d-cv_d.
\]

\(d\) 表示实际交易日，\(w_d\) 是前一日收盘确定的权重，\(r_d\) 是当日日内收益，\(g_d\) 是毛收益，\(v_d\) 是当日双边交易量，\(c\) 是每单位单边成本，\(q_d\) 是净收益。每天平仓，所以即使连续两天权重都是 1，也要每天支付一次进场和一次离场费用；若错用 \(|w_d-w_{d-1}|\)，会漏掉重复日内交易。

这里的固定费率合并了佣金与简化的买卖价差，但没有冲击。**价差（spread）**是可买入卖价与可卖出买价的差，**滑点（slippage）**是实际成交相对参考价的偏离，**冲击（market impact）**是自己的订单改变市场价格的影响。三者可能重叠，真实成本模型应约定记账口径，不能一边重复收取，一边又漏掉大单冲击。

**净值（equity）**将每日收益复利相乘：\(E_d=E_{d-1}(1+q_d)\)，其中初值 \(E_0=1\)。累计收益用最终净值减 1，不能把日收益简单相加当精确复利；当收益足够小时，加总可作近似。**回撤（drawdown）**为当前净值相对此前最高净值的下降比例，它关注资金经历的路径，而不仅是最终结果。


In [ ]:
def replay(weights, realized, cost):
    volume = 2 * np.abs(weights)
    gross = weights * realized
    net = gross - cost * volume
    return {"weight": weights.copy(), "gross": gross, "volume": volume,
            "cost": cost * volume, "net": net}

candidate = replay(weight, actual, contract.cost_per_side)
benchmark = replay(benchmark_weight, actual, contract.cost_per_side)
# 用一笔已知交易确认成本口径：满仓亏 30 bp，再付 2 bp。
example = replay(np.array([1.0]), np.array([-0.003]), contract.cost_per_side)
assert np.isclose(example["net"][0], -0.0032)

train = trade_day < contract.train_end
valid = (trade_day >= contract.train_end) & (trade_day < contract.validation_end)
test = trade_day >= contract.validation_end

def summarize(result, mask):
    net = result["net"][mask]
    equity = np.r_[1.0, np.cumprod(1 + net)]
    peak = np.maximum.accumulate(equity)
    return {
        "days": int(mask.sum()),
        "gross_bp_day": result["gross"][mask].mean() * 10000,
        "net_bp_day": net.mean() * 10000,
        "total_return": equity[-1] - 1,
        "max_drawdown": np.min(equity / peak - 1),
        "volume_day": result["volume"][mask].mean(),
        "participation": np.mean(np.abs(result["weight"][mask]) > 0),
    }

for period, mask in [("训练", train), ("验证", valid), ("测试", test)]:
    for strategy, result in [("候选", candidate), ("每日做多", benchmark)]:
        s = summarize(result, mask)
        print(period, strategy, "天数", s["days"], "净 bp每天", round(s["net_bp_day"], 3),
              "最大回撤 %", round(s["max_drawdown"] * 100, 2))


## 4. 指标要共同回答一个研究判断

平均净收益说明每天平均贡献多少，但不说明不确定性。最终净值说明复利结果，但不说明中途是否经历无法承受的损失。最大回撤补上路径，参与比例说明候选实际暴露多少，交易量说明费用和容量压力。候选若只有一半时间持仓，波动较低可能只是暴露更少，而非择时更聪明。

**夏普比率（Sharpe ratio）**常写成 \(\sqrt K\,\bar q/s_q\)，其中 \(\bar q\) 是每期超额净收益均值，\(s_q\) 是每期标准差，\(K\) 是一年期数。本例现金利率设为零，因此净收益可直接作为超额收益；真实研究应扣除可比资金收益。这个缩放依赖平稳与序列相关等假设；连续性很强、仓位重叠或样本很短时，不能把年化数字当精确证据。本例先报告直接可手算的 bp/日和回撤，不用一个很大的年化比率掩盖只有 100 天测试期的事实。

**样本外（out of sample）**不是数据文件里名为 test 的一段，而是未参与选择的未来样本。本例的规则直接固定，训练与验证帮助观察执行与研究流程；我们没有据它们选择五日窗口。测试只在规则冻结后用于评价。如果随后因为测试不好改成三日、加止损、删除某段，那么那段已经进入开发，不再支持新版本的独立样本外主张。

测试结果还需要和误差范围一起理解。每日收益相关时，可以用**分块重抽样（block bootstrap）**：按连续数日为块抽取收益差，保留一部分短期相关，再估计均值差异的波动范围。块长本身也是选择，短样本无法创造本来没有的信息。合成序列虽然允许知道机制，真实市场无法直接观察“真实延续系数”，所以更需要独立时期与多种证据。

对单资产择时，候选相对每日做多有收益优势，也不直接等于风险调整后的 **alpha**。Alpha 是在约定风险模型下超出风险暴露可解释部分的收益。候选持仓时间与市场状态相关，可能只是减少了某种方向暴露。应同时报告平均权重、与基准的相关性、分状态收益；必要时估计明确的基准回归，并承认尾部与非线性暴露可能仍未解释。


## 5. 压力实验应针对具体脆弱环节

成本翻倍检查“毛收益是否足以容纳估计误差”；额外延迟一天检查“信号是否过快衰减”；分时期检查“结论是否依赖单段关系”；删去最大贡献日检查“证据是否过度集中”。这些是针对本例信息链的诊断，不是为了凑满一张表。

延迟实验尤其容易写错。我们把原先前一日生成的目标再向后移动一交易日，评价的收益日期保持不变。开头没有历史目标的行设空仓，并保留在完整比较集合中。不能把标签也跟着向后移，再误以为增加了执行延迟；那只是重新排列同一配对。

**容量（capacity）**是策略在成本与风险可接受条件下能承载的资金规模。固定 1 bp 费用不能给出容量结论。本例没有成交量、开盘盘口与冲击系数，因此只能说“在假定的小额完全成交下如何”，不能说“可管理某规模资金”。真实延伸要用订单占成交量比例、盘口深度、冲击和资金约束来评估。


In [ ]:
double_cost = replay(weight, actual, 2 * contract.cost_per_side)
delayed_weight = np.r_[0.0, weight[:-1]]
one_day_delay = replay(delayed_weight, actual, contract.cost_per_side)
scenarios = {"基础": candidate, "双倍成本": double_cost, "再延迟一日": one_day_delay}
for scenario, result in scenarios.items():
    s = summarize(result, test)
    print(scenario, "测试净 bp每天", round(s["net_bp_day"], 3),
          "测试回撤 %", round(s["max_drawdown"] * 100, 2))

test_indices = np.flatnonzero(test)
half = len(test_indices) // 2
for block_name, indices in [("测试前半", test_indices[:half]),
                            ("测试后半", test_indices[half:])]:
    print(block_name, "候选净 bp每天", round(candidate["net"][indices].mean() * 10000, 3))

# 固定仓位下，提高非负费率不能提高任何一天净收益。
assert (double_cost["net"] <= candidate["net"] + 1e-15).all()


如何读这段输出？先看候选相对基准的差异，再看压力是否改变研究判断。若双倍成本只是略降但仍保留空间，下一步可以细化成交；若延迟一天后大幅消失，下一步应该测量处理和开盘执行延迟，而不是先增加模型复杂度。若不同半段方向相反，应该问是否关系变化、风险状态变化或少量大日子支配，不能只挑盈利半段当结论。

负面结果也有不同含义。候选毛收益为正而净收益为负，说明当前交易频率和成本不匹配；毛收益本身没有稳定优势，说明输入关系不足；毛收益只来自单个跳空日，而本例并不持有隔夜，首先应排查收益定义错误。把失败定位到信息、预测、持仓或执行层，比笼统说“趋势没用”更有价值。


### 这一次输出到底支持什么

固定样本的测试期中，候选平均净收益约为 5.25 bp/日，每日做多基准约为 8.78 bp/日，因此原先“改善择时”的主假设没有得到这段数据支持。候选虽然自身盈利，但相对基准少了约 3.53 bp/日；它的参与比例仅 56%，最大回撤约 -7.95%，也没有在这段样本中换来更小的最大回撤。结论应先讲这些共同事实，再谈下一步。

双倍费用后候选仍约为 4.13 bp/日，说明在本次假定费用范围里，成本不是唯一失败原因。额外延迟一天反而约为 6.37 bp/日，这不意味着等待更久能提升真实策略，而是短样本差异与弱关系足以改变排序。我们没有据此把额外延迟选为新规则；若想研究它，必须重新提出机制并保留新评价数据。

这一段还暴露了基准的重要性。只把“净收益为正、双倍费用仍为正”放进备忘录，看起来像通过了研究；加上相同条件的每日做多，判断就不同。研究者的责任不是挑几个正确但片面的数字，而是把决定问题的比较放在一起。主假设未获支持，也不等于所有趋势策略都被否定，因为我们只研究一个资产、一个期限和一种持仓规则。

## 6. 从结果走到一页研究备忘录

**研究备忘录（research memo）**的读者可能没有运行你的 Notebook。他需要先知道问题、判断和证据边界，再决定是否投入下一轮数据或工程。开头应直说“当前候选值得继续哪项验证”或“暂不进入资金部署”，随后说明依据。把几十行参数放在第一页，不会自动增加说服力。

一份短备忘录可按连续逻辑组织：研究什么、为什么可能成立；使用什么数据，何时可见；固定了哪些方法和比较对象；独立测试与压力结果怎样；哪些限制最影响结论；下一项实验如何改变决策。这里的限制要具体，例如“没有真实开盘成交和冲击”，比“可能存在风险”有用得多。

下面用当前实际计算值生成正文。它明确区分“教学流程已完成”和“真实机制尚未验证”。即使某个指标碰巧很好，结论也不会突然变成部署建议，因为缺少真实数据和成交证据这个事实没有改变。最终的决策不是由一条 `if Sharpe > 2` 自动取代研究判断。


In [ ]:
s = summarize(candidate, test)
b = summarize(benchmark, test)
stress = summarize(double_cost, test)
lagged = summarize(one_day_delay, test)
relative = s["net_bp_day"] - b["net_bp_day"]
memo = f"""研究判断：当前候选保留为教学研究对象，不进入资金部署。
研究问题：过去五日日内均值为正，是否改善下一日日内做多的择时？
数据与时点：合成日内收益；信号在前一收盘完成，次日开盘进、收盘出。
方法：五日固定窗口，正值满仓做多，否则空仓；每天进出各 1 bp。
独立保留期：{s['days']} 天，候选平均净收益 {s['net_bp_day']:.3f} bp/日，
每日做多基准 {b['net_bp_day']:.3f} bp/日，差异 {relative:.3f} bp/日。
候选最大回撤 {s['max_drawdown']:.2%}，参与比例 {s['participation']:.1%}。
双倍成本后 {stress['net_bp_day']:.3f} bp/日，再延迟一日 {lagged['net_bp_day']:.3f} bp/日。
证据边界：过程使用冻结规则，但样本合成、历史短，无真实开盘成交与容量数据。
下一步：用许可清楚的本地真实开收盘数据复核同一合同，再测成交成本与独立时期。
改变决策的证据：真实保留期出现风险可解释之外的稳定增量，且实际费用未耗尽空间。
"""
print(memo)


上面的“独立保留期”仅指它未参与本章程序内的参数选择，不把合成样本升级成现实证据。若你读完本章后又试了很多窗口，应在自己的记录中写出全部候选与选择过程；最终交付应包含足以重建选择的配置和结果，不能只保留胜出的版本。

**收益归因（PnL attribution）**是在合理粒度上解释盈亏由何而来。本例至少可以分成持仓乘实际收益的毛贡献与交易成本；进一步可看哪些日期贡献、是少数上涨日还是普遍改善、持仓比例如何变化。真实交易还要分隔夜与日内、价差与冲击、方向与风险因子、借券与融资。归因不是给每个亏损编故事，而是让记账项和研究假设对应。

“下一步”也要有优先级。如果没有真实信息时点，先买更强模型没有意义；若没有成本空间，先精炼报告版式也不会改变可交易性。本例最能改变判断的是可交易资产的真实开收盘和成交成本，而不是再遍历 100 个窗口。一个好研究计划会说明什么新证据能让你继续，也说明什么结果会让你停止。


## 7. 交付包要让另一位研究员重建同一笔交易

**可复现（reproducibility）**表示按相同输入与配置能得到相同计算结果；**可重复验证（replication）**还要求换独立数据、时期或实现后检查结论是否成立。固定随机种子只解决前者的一小部分。真实数据修订、包版本、交易日历与成交规则同样会改变结果。

本例的最小交付包应包含：不可变的研究合同；数据字段与时间定义；生成或读取数据的方法及内容指纹；特征和仓位函数；逐日预测、目标、交易量、费用与净收益；分期与压力报告；结论及限制。它们可以很小，不必先建立大型平台。函数显式接收输入和参数，比依赖一个隐藏的全局 Notebook 状态更容易交接。

**数据血缘（data lineage）**是结果到上游对象的对应关系。例如某日净收益可追到权重，权重追到五日均值，均值追到五条日内收益，再追到当时的开收盘数据版本。**幂等（idempotency）**表示同一输入事件重复执行不会重复下单或重复入账。研究回放可以先保证事件主键不重复，生产还要处理订单重试、断线恢复与成交去重。

哈希只能确认字节是否一致，不能确认价格是真的、授权有效或当时可见。运行清单也不能替代研究解释。下面生成一个内存中的交付摘要，说明输入与规则怎样对应；不把临时文件、缓存或个人路径写进教材。


### 把一条“不变量”写成研究语言

**不变量（invariant）**是无论样本如何变化都应成立的关系。本例中，信号日必须早于交易日，权重只能是 0 或 1，固定仓位下提高非负费率不能让净收益增加，空仓日毛收益与交易费用都应为零。这类条件比“最终收益等于某个写死的小数”更值得检查，因为它们保护研究含义，而不是复制当前实现。

边界同样需要解释。前四天没有完整五日历史，应如何处理？本例从第 5 日开始交易，而不是用不足五天的均值偷偷扩大样本；测试首日的信号可以读取测试之前已知的历史，不需要把窗口清空；真实数据出现重复交易日或缺失开盘时，应先停止该行生成目标，不能默默用未来价格补齐。每个处理都应写回数据合同，别让报告与实现各说一种规则。

测试通过只能说明这些约定未被程序违背，不能说明市场关系有效。相反，即使最终策略亏损，只要时间、风险与记账都正确，实验仍可能给出有用的否定证据。把计算正确性和经济有效性分开，能让研究员在面对负结果时继续定位问题，而不是反复“修复”代码直到出现盈利。


In [ ]:
data_hash = hashlib.sha256(intraday.astype("<f8").tobytes()).hexdigest()
config_json = json.dumps(asdict(contract), ensure_ascii=False, sort_keys=True)
config_hash = hashlib.sha256(config_json.encode("utf-8")).hexdigest()
first_test = int(np.flatnonzero(test)[0])
trade_record = {
    "trade_day": int(trade_day[first_test]),
    "signal_day": int(signal_day[first_test]),
    "feature": float(feature[first_test]),
    "weight": float(weight[first_test]),
    "realized_return": float(actual[first_test]),
    "gross_return": float(candidate["gross"][first_test]),
    "cost_return": float(candidate["cost"][first_test]),
    "net_return": float(candidate["net"][first_test]),
}
print("数据指纹:", data_hash[:16], "配置指纹:", config_hash[:16])
print(json.dumps(trade_record, ensure_ascii=False, indent=2))


## 8. 研究候选怎样逐步成为生产候选

**阶段门（stage gate）**表示每个阶段只有回答关键问题，才投入下一阶段资源。假设阶段需要明确机制、期限与可否证条件；数据阶段需要证明时点和版本；信号阶段需要保留期增量；组合阶段需要风险与相关性；执行阶段需要成本和容量；纸面运行需要新数据到来时规则不经人工挑选；最后才讨论生产监控与责任人。

本例只完成了教学数据上的规则、回放、比较和报告。它没有通过真实数据门，也没有真实成交门。因此无论输出多好，都不能跳到生产。**纸面交易（paper trading）**用实时到来的数据生成模拟交易，检查规则是否能按时运行；**影子运行（shadow mode）**让候选与现有系统并行产生决策而不承担真实仓位，便于比较延迟与风险。二者都不能完全替代实际冲击，但能暴露历史回放看不见的运行问题。




![从 100 个想法到 1 个生产候选的 stage gate 漏斗](figures/ch09_stage_gates.png)

**图 9-2｜大多数想法应该早死。** 漏斗从 100 个想法逐层筛到 1 个生产候选，每个 gate 右侧写着通过证据：机制一句话、样本外验证、成本与容量、纸面交易、监控与回滚。漏斗的意义不是淘汰本身，而是让失败发生在占用资金之前。数据类型：概念示意图（比例为示意，不是任何公司的真实漏斗）。


阶段门的数量可以依任务调整，关键是失败发生时知道停在哪一层。若真实数据连开盘价格定义都不一致，不值得先做复杂收益归因；若信号只在不可能的延迟下成立，应在执行门停止，而不是偷偷换成更早成交价。图中漏斗比例只是示意，不代表一家机构的成功率，也不能当研究考核指标。

生产候选还要有**风险限额（risk limit）**、**降级（fallback）**与**紧急停止（kill switch）**。限额明确单名、总仓位、杠杆、损失和订单速度；降级约定特征缺失时是沿用旧值、空仓还是人工处理；紧急停止定义数据陈旧、订单异常或风险突破后如何停止发新单、撤单与平仓。停止发单不自动等于已经平掉持仓，必须区分状态。

**回滚（rollback）**是回到已知版本的规则与配置，不是只把模型文件换回去。模型、缩放参数、特征定义、数据版本与执行约定需要成套对应。还要明确谁可以恢复系统、谁承担异常响应，以及怎样在恢复前核对真实持仓。只做多的日内教材虽然没有这些组件，也应该能够清楚指出缺口。


## 9. 把同一主线带入两个教学项目

项目 A 是跨资产信号与组合研究。不要一开始同时换窗口、换权重、换成本；先选一个信号，用本章的方法追踪一笔交易。第一天上午建立字段字典、价格收益与缺失说明，下午把信号时点、目标仓位、延迟和净值连起来。当天的可观察产物是一张能解释时间线的结果表，而不是尽量多的收益曲线。应亲自回答：“这个收益是日内、隔夜还是连续持有？基准与候选承担了哪些不同风险？”

第二天对同一合同做成本、延迟、时期和资产贡献诊断，再写备忘录。跨资产版本的额外难点在于不同交易日历、币种、波动尺度与期货换月。逆波动权重只调整风险尺度，不自动产生预测能力；某资产贡献极大时，先查风险暴露与数据单位。需要自己决定的一项工程取舍可以是“用共同交易日还是各市场本地时钟”，但决定后要把缺失与成交时点贯穿到底，不能在不同图里换口径。

项目 B 是市场与文本特征的增量研究。第一天先用均值和市场模型，再加词典；亲手为一份公告画发布时间、入库、特征完成与交易时点，确认标签只在事后用于评分。第二天加入结构化事件，或者在本地已具备模型时尝试嵌入，比较语言质量、按时覆盖率与成本后结果。默认模型不可用时就保留未运行状态，不能用假输出填满对比表。

这两个项目不是互相独立的打卡任务。它们共享本章合同、回放与备忘录结构：A 让你理解风险和执行如何改变信号，B 让你理解新增信息如何接受相同规则的增量检验。每个阶段都应留下一个你需要回答的概念问题和一个有后果的工程选择。例如“文本更准却交易更差为什么”是概念问题，“迟到公告设零信号还是推迟全部模型的决策”是工程选择。解释自己的选择，才是项目学习的一部分。


## 10. 同一个项目如何对不同岗位讲清楚

面向**量化研究（quantitative research，QR）**，重点是机制、时间设计、选择偏差、基线和独立证据。本例应解释为什么用每日做多比较、为什么五日窗口提前固定、测试短会怎样限制判断。被问“是不是只是股票上涨”时，应提出匹配暴露、基准回归与分状态诊断，而不是重复净值数字。

面向**量化交易（quantitative trading，QT）**，重点是信号衰减、风险与执行。本例再延迟一日的结果比岭回归公式更值得展开；还要能回答“开盘只成交一半怎么办”“数据迟到后是否补单”“当天如何平仓”。面向**量化开发（quantitative development，QD）**，重点是事件模型、时间对齐、幂等、回放、性能和故障恢复，应该能从那条逐日交易记录解释主键、输入输出和重跑行为。

面向做市岗位，本例没有盘口、队列和库存，必须承认它只能展示研究方法，不能当成做市策略。做市更关心买卖报价、库存风险、逆向选择与对冲；**逆向选择（adverse selection）**指交易对手更可能在对自己不利的信息条件下与你成交。面向组合研究，要强调相关性、共同风险、风险预算和容量；面向基本面或事件研究，要强调披露时点、预期差、修订和原文证据。

公司名字并不能替代职位要求。同一家公司的不同团队可能做高频执行、统计预测或长期配置，应从当年岗位说明和真实研究对象判断准备重点。概率、统计、编程与市场机制会共同出现，但本课程不要求重新背一遍基础概率目录。更有效的准备是把已有数学知识应用到一个你能解释全链条的项目，再针对岗位补充缺口。


## 11. 九十秒表达：每一句都能接住追问

可以这样讲本章候选：“我研究短期日内收益延续，问题是过去五日平均收益能否改善次日日内做多。数据是合成序列，目的是先建立严格的研究流程；前一日收盘算信号，下一日开盘进、收盘平，不计隔夜收益。我先固定五日窗口与 0/1 仓位，和每天日内做多比较，每次进出各收 1 bp。按时间保留测试期，报告实际输出的净收益差、回撤、参与比例，并做双倍成本和额外一天延迟。结果只支持对这份教学流程的判断，没有真实开盘成交与容量证据。下一步是用许可清楚的本地真实数据复核同一合同，再判断是否值得纸面运行。”

口述中的数字应由自己的运行结果替换，不能背一个比实际表现更好的版本。先说明合成数据可能让面试中的收益故事没那么耀眼，却能让对方准确理解你证明了什么。如果有真实实验，再明确区分真实样本与教学实验，不混用两者的证据。

被追问时，不必把整个课程讲一遍。对“为什么五日”，回答选择动机、是否预先固定和未证明最优；对“如何排除泄漏”，拿一笔交易画时间；对“收益来自哪里”，展示毛收益、费用与贡献日；对“为什么不上线”，指出真实数据、成交与运行缺口。这样的回答逐句对应可检查对象，优于堆叠术语。

两周巩固也可以围绕同一个候选循环：一天手算与代码对齐，下一天做一项有机制依据的诊断，再写一页备忘录与九十秒口述。每天保留一点编程和心算训练，但研究时间优先用于能改变判断的实验。准备的质量不是看过多少公司材料，而是面对一个反例时能否定位假设、修正判断并说明下一步。


## 12. 常见误解

“交付就是提交 Notebook。”Notebook 可以承载实验，但交付还需要数据含义、冻结配置、逐笔对象与研究结论，别人才能接手。“限制写得越少越有说服力。”没有真实成交却回避容量问题，会让主要证据失去可信度。“最高指标就是最终答案。”窗口、阈值与样本若反复挑选，最高值常反映搜索过程，不能代替新样本。

“所有阶段都过了才算有成果。”定位某种信号在成本门失败，也是明确成果，可以节省后续资源。“让程序生成报告就不需要判断。”程序能填入实际指标，却不能凭一个阈值决定合成关系是否适用于现实。“按岗位改表达就是换个好听故事。”真正的调整是突出对方负责的对象，同时保持研究合同、数据和结论完全一致。


## 13. 手工练习

### 练习一：从五天历史追踪到净收益

已知前五日日内收益为 +20、-10、+15、-5、+10 bp，接下来两日分别为 -30、+25 bp。按五日均值正则下一日做多，进出各 1 bp，计算这两天仓位与总净收益。第二天信号计算时必须更新窗口。

<details><summary>参考答案</summary>

第一天前五日和为 30 bp，均值 +6 bp，做多，净收益 \(-30-2=-32\) bp。第二天窗口移除最早的 +20，加入 -30，五日和变为 -20 bp，均值 -4 bp，因此空仓，净收益 0。两日净收益加总 -32 bp，此时仅第一天有交易，精确复利也为 -0.32%。若一直保留原五天窗口，会在第二天错误做多；若用第二天的 +25 决定当天仓位，则是未来信息泄漏。

</details>

### 练习二：决定下一项实验

某候选测试毛收益为 3 bp/日，成本后为 0.5 bp/日，双倍费用后 -2 bp/日；再延迟一天毛收益接近 0。研究员建议立刻试十种神经网络。请写一个更直接的下一步，并给出停止条件。

<details><summary>参考答案</summary>

先核对实际处理时间与可成交价格，细化单位成本与容量，因为现有结果同时对费用和延迟脆弱。可以在不下真实订单的影子运行中记录信号完成、参考报价与模拟成交，再与同一基准比较。若可达到的延迟与成本已经耗尽毛空间，停止当前交易设定；若仍保留空间，再用新的独立时期评估。复杂模型可能值得研究，但当前最影响决策的证据来自执行，不是模型数量。

</details>

### 练习三：把夸张结论改成可复核判断

原文：“策略测试收益很高，证明趋势有效，未来可扩展到大规模资金。”已知数据为合成，测试 100 天，没有成交量。改写成两三句话，保留有价值的信息。

<details><summary>参考答案</summary>

“在冻结规则的 100 天合成保留期中，候选得到所报告的收益、回撤与成本敏感性；它说明研究管线能够一致地处理信息、仓位和费用。数据不支持真实趋势机制或容量结论。下一步使用许可清楚的真实开收盘和成交量数据，在新的保留期复核相同合同，再评估成本与容量。”实际提交时应填入具体计算值及对应基准。

</details>

## 14. 自测与思考

1. 候选只持有一半交易日，波动小于每日做多，能否据此称为风险调整后的 alpha？
2. 为什么每天收盘平仓时，连续两日权重都是 1 仍然需要每天付费？
3. 测试不好后改参数，同一段数据还能怎样使用？
4. 相同种子能复现结果，为何还需要数据版本与配置指纹？
5. 纸面运行成功之后，还缺哪些生产证据？

<details><summary>参考答案</summary>

第一，不能，低波动可能来自较低暴露，需要与匹配风险基准及明确风险模型比较。第二，每天从空仓进场再平仓，目标相同并不表示仓位跨夜保留。第三，它可作为开发或诊断样本，但新版本需要新的独立保留期。第四，种子不固定真实数据修订、字段含义、成本、日历、代码和依赖实现。第五，还需真实成交与冲击、容量、风险限额、订单和持仓一致性、异常恢复、监控、回滚及责任安排；模拟不能保证这些都成立。

</details>

## 15. 本章小结

同一条五日日内趋势规则，连接了研究合同、信息时间、仓位、双边成本、独立比较、压力诊断、备忘录、交付对象和岗位表达。它的简单让任何人都能沿一笔交易找到原始输入，也让限制无法被复杂模型掩盖。

完成量化研究，不是把一个想法讲成必胜故事，而是让下一位研究员知道哪些证据已经具备、哪些假设仍未成立，以及哪个新实验最可能改变判断。能清楚交代继续、缩小、暂停或停止的理由，就是完整研究能力的一部分。
